# Name: Nabiha Sadaf

# Course: MSCS 634
# Lab 6 – Association Rule Mining Using Apriori and FP-Growth

In [ ]:
!pip -q install mlxtend openpyxl

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules


In [ ]:
url='https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
df=pd.read_excel(url)
df.head()

In [ ]:
print(df.shape)
df=df.dropna(subset=['Description'])
df=df[~df['InvoiceNo'].astype(str).str.contains('C')]
df=df[df['Quantity']>0]
print(df.shape)

In [ ]:
item_counts=df['Description'].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=item_counts.values,y=item_counts.index)
plt.title('Top 10 Most Purchased Items')
plt.show()

In [ ]:
basket=(df.groupby(['InvoiceNo','Description'])['Quantity']
          .sum().unstack().fillna(0)>0)
print(basket.shape)
basket.head()

In [ ]:
top_items=item_counts.index
heat=basket[top_items].astype(int).T.dot(basket[top_items].astype(int))
plt.figure(figsize=(10,8))
sns.heatmap(heat,cmap='Blues')
plt.title('Item Co-occurrence Heatmap')
plt.show()

In [ ]:
start=time.time()
apriori_sets=apriori(basket,min_support=0.02,use_colnames=True)
apriori_time=time.time()-start
print(apriori_time)
apriori_sets.head()

In [ ]:
top=apriori_sets.sort_values('support',ascending=False).head(10).copy()
top['itemsets']=top['itemsets'].astype(str)
plt.figure(figsize=(10,5))
sns.barplot(data=top,x='support',y='itemsets')
plt.title('Top Apriori Itemsets')
plt.show()

In [ ]:
start=time.time()
fp_sets=fpgrowth(basket,min_support=0.02,use_colnames=True)
fp_time=time.time()-start
print(fp_time)
fp_sets.head()

In [ ]:
rules=association_rules(fp_sets,metric='confidence',min_threshold=0.5)
rules[['antecedents','consequents','support','confidence','lift']].head(10)

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=rules,x='confidence',y='lift',size='support')
plt.title('Confidence vs Lift')
plt.show()

print('Apriori Time:',apriori_time)
print('FP-Growth Time:',fp_time)

## Comparative Analysis

- FP-Growth was faster because it uses an FP-tree instead of generating candidate itemsets repeatedly.
- Data cleaning removed missing descriptions, cancelled invoices, and negative quantities.
- Association rules identified products that are commonly purchased together.
